In [0]:
table_name = "dim_plan_types"
silver_table_fqn = f"he_prod_incen_analytics_dbw_01.silver.{table_name}"

df = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_plan_types")

In [0]:
from pyspark.sql.functions import col, upper, trim, when, abs as spark_abs

# 1. Standardize Text
df = df.withColumn("plan_type_id", upper(trim(col("plan_type_id"))))
df = df.withColumn("plan_type_name", upper(trim(col("plan_type_name"))))
df = df.withColumn("network_requirement", upper(trim(col("network_requirement"))))

# 2. Precision Optimization (Safe Cast Decimals - handles "Unlimited" gracefully)
for dec_col in ["sum_insured_min", "sum_insured_max", "room_rent_limit"]:
    is_valid = trim(col(dec_col)).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
    df = df.withColumn(dec_col, when(is_valid, trim(col(dec_col)).cast("decimal(15,2)")).otherwise(None))
    df = df.withColumn(dec_col, spark_abs(col(dec_col)))

for dec_col in ["copay_percentage_min", "copay_percentage_max"]:
    is_valid = trim(col(dec_col)).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
    df = df.withColumn(dec_col, when(is_valid, trim(col(dec_col)).cast("decimal(5,2)")).otherwise(None))

# 3. Safe Cast Integers
for int_col in ["pre_hospitalization_days", "post_hospitalization_days"]:
    df = df.withColumn(int_col, trim(col(int_col)).cast("int"))

# 4. Boolean Mapping
for bool_col in ["maternity_coverage", "critical_illness_covered"]:
    df = df.withColumn(bool_col, 
                       when(upper(trim(col(bool_col))).isin("YES", "1", "TRUE"), True)
                       .when(upper(trim(col(bool_col))).isin("NO", "0", "FALSE"), False)
                       .otherwise(None).cast("boolean"))

# 5. Drop audit/source SKs
df = df.drop("_ingested_at", "_source_file", "plan_type_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_fqn)
print(f"✅ Successfully wrote {table_name} to Silver layer.")